# Imports

In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
from pettingzoo.utils import wrappers
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from pettingzoo.atari import surround_v2
import numpy as np
import matplotlib.pyplot as plt
import cv2
from datetime import datetime
import ale_py
import multi_agent_ale_py


: 

# Env Setup

In [ ]:
# Define a wrapper for Gym env to add survival rewards
class RewardWrapper(gym.Wrapper):
    def __init__(self, env, survival_reward=0.01):
        super(RewardWrapper, self).__init__(env)
        self.survival_reward = survival_reward

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # Add a small positive reward for surviving
        reward += self.survival_reward
        return obs, reward, terminated, truncated, info

In [ ]:
import torch 
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticCnnPolicy

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.skip_connection = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip_connection = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.skip_connection(x)
        out = self.relu(out)
        return out

class ResNetCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512):
        super(ResNetCNN, self).__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.conv1 = nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4, padding=0)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        
        self.layer1 = self._make_layer(32, 64, stride=2)
        self.layer2 = self._make_layer(64, 128, stride=2)
        self.layer3 = self._make_layer(128, 128, stride=2)
        self.layer4 = self._make_layer(128, 128, stride=2)
        
        self.flatten = nn.Flatten()
        
        # Compute shape by doing one forward pass
        with torch.no_grad():
            n_flatten = self._forward_conv(torch.as_tensor(observation_space.sample()[None]).float()).shape[1]
            
        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def _make_layer(self, in_channels, out_channels, stride):
        return nn.Sequential(
            ResidualBlock(in_channels, out_channels, stride),
            ResidualBlock(out_channels, out_channels)
        )

    def _forward_conv(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.flatten(x)
        return x

    def forward(self, observations):
        conv_out = self._forward_conv(observations)
        return self.linear(self.flatten(conv_out))

class CustomResNetPolicy(ActorCriticCnnPolicy):
    def __init__(self, *args, **kwargs):
        super(CustomResNetPolicy, self).__init__(*args, **kwargs, features_extractor_class=ResNetCNN, features_extractor_kwargs=dict(features_dim=512))


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
import datetime
from gymnasium.wrappers import RecordVideo

def train_gym_agent(render_game_every_n=5, render_during_training=True):
    """
    Train the agent with optional rendering.
    """
    # Get the current timestamp
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # Create the Gymnasium environment
    env = gym.make("ALE/Surround-v5", render_mode="rgb_array", mode=2, difficulty=1)
    env = RewardWrapper(env)

    # Wrap the environment with VecMonitor to log episode rewards
    # vec_env = DummyVecEnv([lambda: env])
    # vec_env = VecMonitor(vec_env, f"./logs_{timestamp}/")

    # Wrap the environment with RecordVideo to record videos
    video_folder = f"./videos_{timestamp}/"
    env = RecordVideo(env, video_folder=video_folder, episode_trigger=lambda episode_id: episode_id % render_game_every_n == 0)

    # Create the model
    model = PPO("CnnPolicy", env, verbose=1, tensorboard_log=f"./ppo_surround_{timestamp}/")

    # Create an evaluation callback with TensorBoard logging
    eval_callback = EvalCallback(env, best_model_save_path=f'./logs_{timestamp}/best_model',
                                 log_path=f'./logs_{timestamp}/', eval_freq=500,
                                 deterministic=True, render=False)

    # Train the model
    model.learn(total_timesteps=1000000, callback=eval_callback)

    # Save the model
    model.save(f"ppo_surround_agent_{timestamp}")

    # Optionally, load the model
    # model = PPO.load(f"ppo_surround_agent_{timestamp}")

# Call the function to train the agent
train_gym_agent(render_game_every_n=5, render_during_training=True)

# Petting Zoo


In [ ]:

def test_pettingzoo_selfplay(render_game_every_n=5, track_actions=True, video_filename="gameplay.mp4"):
    # Load the trained Gymnasium agent
    model = PPO.load("ppo_surround_agent")

    # Create the PettingZoo environment with the render_mode set to "rgb_array" (so we can access frames)
    env = DummyVecEnv([lambda: surround_v2.env(render_mode="rgb_array")])
    env.reset()
    
    # Get the first observation for the agent
    first_agent = next(iter(env.agent_iter()))  # Get the first agent
    obs, *_ = env.last()  # Get the observation for that agent
    frame_height, frame_width, _ = obs.shape

    # plot the first observation
    obs = cv2.cvtColor(obs, cv2.COLOR_BGR2RGB)
    plt.imshow(obs)

    # Initialize OpenCV VideoWriter to write the frames to a video file
    fourcc = cv2.VideoWriter_fourcc(*'H264')  # Codec for .mp4 file
    video_writer = cv2.VideoWriter(video_filename, fourcc, 30, (frame_width, frame_height))  # 30 FPS

    env.reset()
    # Initialize tracking variables
    actions_taken = {agent: [] for agent in env.agents}  # Dictionary to track actions per agent
    frame_count = 0  # Counter for frames rendered

    # Run self-play in the PettingZoo environment
    for agent in env.agent_iter():
        result = env.last()  # Get the result of the current step
        
        obs, reward, terminated, truncated, info = result

        if not terminated and not truncated:
            # Use the trained model for all agents
            action = model.predict(obs, deterministic=True)[0]
            env.step(action)
            
            # Track actions taken
            if track_actions:
                actions_taken[agent].append(action)

        else:
            # Optionally handle the end of the game for that agent
            print(f"Agent {agent} has finished.")
            print("The game is over or truncated.")
            break  

        # Render the game every `render_game_every_n` frames
        frame_count += 1
        render_and_save_video(obs, video_writer, render_game_every_n, frame_count)

    # Close the environment and video writer
    env.close()
    video_writer.release()  # Save and close the video file
    
    print(f"Video saved as {video_filename}")